# Sphere Octant Division With Area Optimizer (`N^2`)

Divide the first sphere octant (`x, y, z >= 0`) into `N²` spherical triangles and visualize them.

- The mesh is built from a simplex lattice and normalized onto the sphere.
- Visualization uses `matplotlib` 3D plotting.
- The last cell checks side-length patterns.


In [ ]:
import numpy as np


def lattice_to_octant_point(i, j, k, N):
    ring_ij, ring_jk, ring_ki = i + j, j + k, k + i

    theta_ij = (np.pi * ring_ij) / (2.0 * N)
    theta_jk = (np.pi * ring_jk) / (2.0 * N)
    theta_ki = (np.pi * ring_ki) / (2.0 * N)
    
    phi_ij = 0.0 if ring_ij == 0 else (np.pi * j) / (2.0 * ring_ij)
    phi_jk = 0.0 if ring_jk == 0 else (np.pi * k) / (2.0 * ring_jk)
    phi_ki = 0.0 if ring_ki == 0 else (np.pi * i) / (2.0 * ring_ki)

    return np.array([
        np.sin(theta_ij) * np.cos(phi_ij) + np.sin(theta_ki) * np.sin(phi_ki) + np.cos(theta_jk),
        np.sin(theta_jk) * np.cos(phi_jk) + np.sin(theta_ij) * np.sin(phi_ij) + np.cos(theta_ki),
        np.sin(theta_ki) * np.cos(phi_ki) + np.sin(theta_jk) * np.sin(phi_jk) + np.cos(theta_ij),
    ], dtype=float) / 3


def build_octant_points(N):
    if N < 1:
        raise ValueError('N must be >= 1.')

    points = {}
    for i in range(N + 1):
        for j in range(N + 1 - i):
            k = N - i - j
            points[(i, j, k)] = lattice_to_octant_point(i, j, k, N)

    return points


def build_octant_triangle_keys(N):
    if N < 1:
        raise ValueError('N must be >= 1.')

    triangles = []
    for i in range(N):
        for j in range(N - i):
            k = N - i - j
            a = (i, j, k)
            b = (i + 1, j, k - 1)
            c = (i, j + 1, k - 1)
            triangles.append((a, b, c))

            if k >= 2:
                d = (i + 1, j + 1, k - 2)
                triangles.append((b, d, c))
    return triangles


def build_octant_mesh(N):
    points = build_octant_points(N)
    triangle_keys = build_octant_triangle_keys(N)
    tri_xyz = [np.array([points[idx] for idx in tri]) for tri in triangle_keys]
    return points, triangle_keys, tri_xyz


def build_point_index(points):
    point_keys = sorted(points.keys(), key=lambda t: (t[0] + t[1], t[0], t[1], t[2]))
    point_index = {key: idx for idx, key in enumerate(point_keys)}
    return point_keys, point_index


def triangle_side_lengths(tri):
    def ang(u, v):
        return np.arccos(np.clip(np.dot(u, v), -1.0, 1.0))

    a, b, c = tri
    return np.array([ang(a, b), ang(b, c), ang(c, a)])


## Deterministic tension iterator for equal spherical areas

Start from the existing octant mesh and iteratively move vertices so spherical triangle areas approach the global mean.

In [ ]:
# Deterministic iteration to equalize spherical triangle areas

import matplotlib.pyplot as plt
from sphere_geometry_util import geodesic_arc, normalize


def spherical_triangle_area(a, b, c):
    # Unit-sphere area from spherical excess (robust atan2 form)
    a = normalize(a)
    b = normalize(b)
    c = normalize(c)
    det = abs(np.dot(a, np.cross(b, c)))
    denom = 1.0 + np.dot(a, b) + np.dot(b, c) + np.dot(c, a)
    return 2.0 * np.arctan2(det, max(denom, 1e-15))


def classify_vertex_constraint(key, N):
    i, j, k = key
    zeros = [i == 0, j == 0, k == 0]
    zc = sum(zeros)

    if zc >= 2:
        # Corner vertices are fixed axis endpoints
        return ('corner', None)
    if i == 0:
        return ('edge', 0)  # x=0 edge arc
    if j == 0:
        return ('edge', 1)  # y=0 edge arc
    if k == 0:
        return ('edge', 2)  # z=0 edge arc
    return ('interior', None)


def project_vertex(v, key, N):
    mode, axis = classify_vertex_constraint(key, N)

    # Fixed corners
    if mode == 'corner':
        i, j, k = key
        if i == N:
            return np.array([1.0, 0.0, 0.0])
        if j == N:
            return np.array([0.0, 1.0, 0.0])
        return np.array([0.0, 0.0, 1.0])

    v = np.asarray(v, dtype=float).copy()
    v = np.maximum(v, 0.0)  # keep octant

    if mode == 'edge':
        v[axis] = 0.0
        # Keep on great-circle arc in the octant plane
        free = [0, 1, 2]
        free.remove(axis)
        if v[free[0]] == 0.0 and v[free[1]] == 0.0:
            v[free] = 1.0 / np.sqrt(2.0)

    n = np.linalg.norm(v)
    if n <= 0.0:
        # Fallback should be rare; recover from initial lattice position
        v = normalize(lattice_to_octant_point(*key, N))
        mode2, axis2 = classify_vertex_constraint(key, N)
        if mode2 == 'edge':
            v[axis2] = 0.0
            v = normalize(np.maximum(v, 0.0))
        return v

    return v / n


def project_center_for_vertex(center, key, N):
    mode, axis = classify_vertex_constraint(key, N)
    c = normalize(center)
    if mode == 'corner':
        return project_vertex(c, key, N)
    if mode == 'edge':
        c = np.maximum(c, 0.0)
        c[axis] = 0.0
        return normalize(c)
    return c


def run_tension_equalizer(N, iterations=300, lr=0.18, verbose_every=25):
    points0, triangle_keys, _ = build_octant_mesh(N)
    point_keys = sorted(points0.keys(), key=lambda t: (t[0] + t[1], t[0], t[1], t[2]))

    # Initial value: normalized result from sphere_octant_division.ipynb
    positions = {k: project_vertex(normalize(points0[k]), k, N) for k in point_keys}

    incident = {k: [] for k in point_keys}
    for t_id, tri in enumerate(triangle_keys):
        for vk in tri:
            incident[vk].append(t_id)

    history = []

    for it in range(1, iterations + 1):
        tri_areas = np.empty(len(triangle_keys), dtype=float)
        tri_centers = []

        for t_id, tri in enumerate(triangle_keys):
            a, b, c = [positions[k] for k in tri]
            tri_areas[t_id] = spherical_triangle_area(a, b, c)
            tri_centers.append(normalize(a + b + c))

        mean_area = float(tri_areas.mean())
        std_area = float(tri_areas.std(ddof=0))
        max_rel = float(np.max(np.abs(tri_areas - mean_area) / max(mean_area, 1e-15)))
        history.append((it, mean_area, std_area, max_rel))

        if verbose_every and (it % verbose_every == 0 or it == iterations - 1):
            print(f'iter={it:4d} mean={mean_area:.8e} std={std_area:.8e} max_rel={max_rel:.4e}')

        if it == iterations:
            break

        proposals = {k: [] for k in point_keys}

        for t_id, tri in enumerate(triangle_keys):
            area = tri_areas[t_id]
            rel = (area - mean_area) / max(mean_area, 1e-15)
            center = tri_centers[t_id]

            for vk in tri:
                v = positions[vk]
                c = project_center_for_vertex(center, vk, N)
                delta = lr * rel * (c - v)
                proposals[vk].append(delta)

        new_positions = {}
        for k in point_keys:
            if proposals[k]:
                move = np.mean(np.vstack(proposals[k]), axis=0)
            else:
                move = np.zeros(3, dtype=float)
            new_positions[k] = project_vertex(positions[k] + move, k, N)

        positions = new_positions

    return positions, triangle_keys, np.array(history, dtype=float)


In [ ]:
# Run iterator
N = 16
positions_eq, triangle_keys_eq, hist = run_tension_equalizer(
    N, iterations=1000, lr=0.1, verbose_every=50
)

# Final spherical area distribution
areas_eq = []
for tri in triangle_keys_eq:
    a, b, c = [positions_eq[k] for k in tri]
    areas_eq.append(spherical_triangle_area(a, b, c))
areas_eq = np.array(areas_eq, dtype=float)

print('\n=== Spherical Triangle Area Distribution (after tension iteration) ===')
print(f'triangle_count = {areas_eq.size}')
print(f'min   = {areas_eq.min():.10f}')
print(f'max   = {areas_eq.max():.10f}')
print(f'mean  = {areas_eq.mean():.10f}')
print(f'median= {np.median(areas_eq):.10f}')
print(f'std   = {areas_eq.std(ddof=0):.10f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
line_std = ax1.plot(hist[:, 0], hist[:, 2], color='tab:blue', lw=1.8, label='std')[0]
ax1.set_title('Convergence history')
ax1.set_xlabel('iteration')
ax1.set_ylabel('std', color='tab:blue')
ax1.set_ylim([0, np.max(hist[:, 2])*1.1])
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(alpha=0.3)

ax1r = ax1.twinx()
line_max = ax1r.plot(hist[:, 0], hist[:, 3], color='tab:red', lw=1.8, label='max_rel_dev')[0]
ax1r.set_ylabel('max_rel_dev', color='tab:red')
ax1r.set_ylim([0, np.max(hist[:, 3])*1.1])
ax1r.tick_params(axis='y', labelcolor='tab:red')
ax1.legend([line_std, line_max], ['std', 'max_rel_dev'], loc='upper right')

bins = min(20, max(5, int(np.sqrt(areas_eq.size))))
ax2.hist(areas_eq, bins=bins, color='teal', edgecolor='black', alpha=0.85)
ax2.axvline(areas_eq.mean(), color='crimson', linestyle='--', linewidth=1.5, label=f'mean={areas_eq.mean():.6f}')
ax2.set_title(f'Spherical area distribution (N={N})')
ax2.set_xlabel('spherical area')
ax2.set_ylabel('count')
ax2.grid(alpha=0.25)
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Before/after spherical triangle visualization
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

positions_before = {k: project_vertex(v, k, N) for k, v in build_octant_mesh(N)[0].items()}

tris_before = [np.array([positions_before[k] for k in tri]) for tri in triangle_keys_eq]
tris_after = [np.array([positions_eq[k] for k in tri]) for tri in triangle_keys_eq]

fig = plt.figure(figsize=(14, 6))
ax_l = fig.add_subplot(121, projection='3d')
ax_r = fig.add_subplot(122, projection='3d')

def draw_octant_mesh(ax, tris, title):
    th = np.linspace(0.0, np.pi / 2.0, 50)
    ph = np.linspace(0.0, np.pi / 2.0, 50)
    TH, PH = np.meshgrid(th, ph)
    X = np.sin(TH) * np.cos(PH)
    Y = np.sin(TH) * np.sin(PH)
    Z = np.cos(TH)
    ax.plot_surface(X, Y, Z, color='lightsteelblue', alpha=0.18, linewidth=0, antialiased=True)

    poly = Poly3DCollection(tris, facecolors='cornflowerblue', edgecolors='none', alpha=0.28)
    ax.add_collection3d(poly)

    for tri in tris:
        for e0, e1 in ((0, 1), (1, 2), (2, 0)):
            arc = geodesic_arc(tri[e0], tri[e1], samples=16)
            ax.plot(arc[:, 0], arc[:, 1], arc[:, 2], color='k', linewidth=0.6, alpha=0.7)

    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.set_zlim(0, 1.05)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=28, azim=38)
    ax.set_title(title)

draw_octant_mesh(ax_l, tris_before, f'Before optimization (N={N})')
draw_octant_mesh(ax_r, tris_after, f'After optimization (N={N})')

plt.tight_layout()
plt.show()


In [ ]:
# Coordinate points (text) + permutation error check
import itertools

point_keys = sorted(positions_eq.keys(), key=lambda t: (t[0] + t[1], t[0], t[1], t[2]))
point_index = {k: i for i, k in enumerate(point_keys)}

print('\n=== Points AFTER optimization (index: lattice_index -> [x, y, z]) ===')
for key in point_keys:
    idx = point_index[key]
    x, y, z = positions_eq[key]
    print(f'{idx:3d}: {key} -> [{x:.8f}, {y:.8f}, {z:.8f}]')

print(f'\npoint_count = {len(point_keys)}')

# Permutation-equivariance check for optimized positions
perms = list(itertools.permutations([0, 1, 2]))
max_perm_err = 0.0
worst_case = None

for key in point_keys:
    base = positions_eq[key]
    idx = np.array(key, dtype=int)

    for p in perms:
        kp_arr = idx[list(p)]
        kp = (int(kp_arr[0]), int(kp_arr[1]), int(kp_arr[2]))
        lhs = positions_eq[kp]
        rhs = base[list(p)]
        err = float(np.linalg.norm(lhs - rhs))

        if err > max_perm_err:
            max_perm_err = err
            worst_case = (key, p)

print('\n=== Permutation Equivariance (optimized positions) ===')
print(f'max_permutation_error = {max_perm_err:.3e}')
print('worst_case =', worst_case)
